# 2. 데이터 라벨링 편향 평가 (ML)

## 개요

이 노트북은 머신러닝 데이터셋의 **라벨링 편향(Labeling Bias)**을 평가합니다.

### 평가 기준: 4/5 Rule (80% Rule)

**4/5 Rule**은 고용 차별 판단에 사용되는 통계적 기준으로, 보호 변수(성별, 인종, 나이 등)에 따라 긍정 라벨의 비율 차이가 80% 이상이어야 공정하다고 판단합니다.

**공식:**
```
min(긍정 라벨 비율) / max(긍정 라벨 비율) >= 0.8
```

- **Pass**: 비율이 80% 이상 → 공정함
- **Fail**: 비율이 80% 미만 → 편향 존재

### 데이터셋
- UCI Adult Income Dataset
- 보호 변수: 성별(sex), 교육 수준(education)
- 라벨: 연소득 >50K 여부

## 1. 환경 설정 및 데이터 로드

In [ ]:
# 필요한 라이브러리 설치 (Colab/로컬 환경)
!pip install -q pandas numpy matplotlib seaborn scikit-learn ucimlrepo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 시각화 스타일
sns.set_style('whitegrid')
sns.set_palette('Set2')

print("라이브러리 로드 완료")

In [ ]:
# UCI Adult Income 데이터셋 로드
adult = fetch_ucirepo(id=2)
X = adult.data.features
y = adult.data.targets

# 데이터 결합
df = pd.concat([X, y], axis=1)

# 라벨을 이진 변수로 변환 (>50K = 1, <=50K = 0)
df['income_binary'] = (df['income'] == '>50K').astype(int)

print(f"데이터셋 크기: {df.shape}")
print(f"\n라벨 분포:\n{df['income_binary'].value_counts()}")
df.head()

## 2. 데이터 전처리 및 탐색

In [ ]:
# 결측치 처리
df = df.dropna()

# 보호 변수 확인
print("보호 변수 - 성별(sex) 분포:")
print(df['sex'].value_counts())
print("\n보호 변수 - 교육 수준(education) 분포:")
print(df['education'].value_counts())

# 교육 수준을 High/Low로 단순화 (분석 편의)
high_education = ['Bachelors', 'Masters', 'Doctorate', 'Prof-school']
df['education_level'] = df['education'].apply(
    lambda x: 'High' if x in high_education else 'Low'
)

print("\n교육 수준 그룹화:")
print(df['education_level'].value_counts())

## 3. 라벨링 편향 평가 - 4/5 Rule

In [ ]:
def calculate_labeling_bias(df, protected_attr, label_col='income_binary'):
    """
    4/5 Rule을 사용한 라벨링 편향 계산
    
    Args:
        df: 데이터프레임
        protected_attr: 보호 변수 컬럼명
        label_col: 라벨 컬럼명
    
    Returns:
        dict: 그룹별 긍정 라벨 비율, 4/5 Rule 비율, Pass 여부
    """
    # 그룹별 긍정 라벨 비율 계산
    positive_rates = df.groupby(protected_attr)[label_col].mean()
    
    # 4/5 Rule 계산
    min_rate = positive_rates.min()
    max_rate = positive_rates.max()
    four_fifths_ratio = min_rate / max_rate if max_rate > 0 else 0
    
    # 80% 기준 통과 여부
    is_fair = four_fifths_ratio >= 0.8
    
    return {
        'positive_rates': positive_rates,
        'four_fifths_ratio': four_fifths_ratio,
        'is_fair': is_fair,
        'min_rate': min_rate,
        'max_rate': max_rate
    }

print("=" * 60)
print("라벨링 편향 평가 (4/5 Rule)")
print("=" * 60)

### 3.1 성별(Sex)에 따른 라벨링 편향

In [ ]:
# 성별에 따른 라벨링 편향 평가
sex_bias = calculate_labeling_bias(df, 'sex')

print("\n[성별(Sex) 분석 결과]")
print("\n그룹별 긍정 라벨 비율 (소득 >50K):")
for group, rate in sex_bias['positive_rates'].items():
    print(f"  {group}: {rate:.4f} ({rate*100:.2f}%)")

print(f"\n4/5 Rule 비율: {sex_bias['four_fifths_ratio']:.4f} ({sex_bias['four_fifths_ratio']*100:.2f}%)")
print(f"80% 기준: {'PASS ✓' if sex_bias['is_fair'] else 'FAIL ✗'}")

if not sex_bias['is_fair']:
    disparity = (1 - sex_bias['four_fifths_ratio']) * 100
    print(f"\n⚠️ 편향 발견: 최소/최대 비율 차이 {disparity:.2f}% (기준: 20% 이내)")

### 3.2 교육 수준(Education Level)에 따른 라벨링 편향

In [ ]:
# 교육 수준에 따른 라벨링 편향 평가
edu_bias = calculate_labeling_bias(df, 'education_level')

print("\n[교육 수준(Education Level) 분석 결과]")
print("\n그룹별 긍정 라벨 비율 (소득 >50K):")
for group, rate in edu_bias['positive_rates'].items():
    print(f"  {group}: {rate:.4f} ({rate*100:.2f}%)")

print(f"\n4/5 Rule 비율: {edu_bias['four_fifths_ratio']:.4f} ({edu_bias['four_fifths_ratio']*100:.2f}%)")
print(f"80% 기준: {'PASS ✓' if edu_bias['is_fair'] else 'FAIL ✗'}")

if not edu_bias['is_fair']:
    disparity = (1 - edu_bias['four_fifths_ratio']) * 100
    print(f"\n⚠️ 편향 발견: 최소/최대 비율 차이 {disparity:.2f}% (기준: 20% 이내)")

## 4. 결과 시각화

In [ ]:
# 시각화 1: 그룹별 긍정 라벨 비율 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 성별
sex_rates = sex_bias['positive_rates']
colors_sex = ['#2ecc71' if sex_bias['is_fair'] else '#e74c3c'] * len(sex_rates)
axes[0].bar(sex_rates.index, sex_rates.values, color=colors_sex, alpha=0.8, edgecolor='black')
axes[0].axhline(y=0.8 * sex_bias['max_rate'], color='red', linestyle='--', 
                label=f'80% Threshold ({0.8 * sex_bias["max_rate"]:.3f})')
axes[0].set_title('Positive Label Rate by Sex', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Positive Label Rate (Income >50K)', fontsize=11)
axes[0].set_xlabel('Sex', fontsize=11)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 값 표시
for i, (idx, val) in enumerate(sex_rates.items()):
    axes[0].text(i, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

# 교육 수준
edu_rates = edu_bias['positive_rates']
colors_edu = ['#2ecc71' if edu_bias['is_fair'] else '#e74c3c'] * len(edu_rates)
axes[1].bar(edu_rates.index, edu_rates.values, color=colors_edu, alpha=0.8, edgecolor='black')
axes[1].axhline(y=0.8 * edu_bias['max_rate'], color='red', linestyle='--', 
                label=f'80% Threshold ({0.8 * edu_bias["max_rate"]:.3f})')
axes[1].set_title('Positive Label Rate by Education Level', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Positive Label Rate (Income >50K)', fontsize=11)
axes[1].set_xlabel('Education Level', fontsize=11)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# 값 표시
for i, (idx, val) in enumerate(edu_rates.items()):
    axes[1].text(i, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/labeling_bias_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 2: 4/5 Rule 비율 비교
fig, ax = plt.subplots(figsize=(10, 6))

attributes = ['Sex', 'Education Level']
ratios = [sex_bias['four_fifths_ratio'], edu_bias['four_fifths_ratio']]
colors = ['#2ecc71' if r >= 0.8 else '#e74c3c' for r in ratios]

bars = ax.barh(attributes, ratios, color=colors, alpha=0.8, edgecolor='black')
ax.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='80% Threshold')
ax.set_xlabel('4/5 Rule Ratio', fontsize=12, fontweight='bold')
ax.set_title('Labeling Bias Evaluation - 4/5 Rule', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.0)
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)

# 값과 상태 표시
for i, (attr, ratio) in enumerate(zip(attributes, ratios)):
    status = 'PASS ✓' if ratio >= 0.8 else 'FAIL ✗'
    ax.text(ratio + 0.02, i, f'{ratio:.3f} ({status})', 
            va='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('../assets/four_fifths_rule_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 3: 교차 분석 (Sex x Education Level)
cross_table = df.groupby(['sex', 'education_level'])['income_binary'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 6))
cross_table.plot(kind='bar', ax=ax, color=['#3498db', '#e67e22'], alpha=0.8, edgecolor='black')
ax.set_title('Positive Label Rate by Sex and Education Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Positive Label Rate (Income >50K)', fontsize=11)
ax.set_xlabel('Sex', fontsize=11)
ax.legend(title='Education Level', loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(axis='y', alpha=0.3)

# 값 표시
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/cross_analysis_sex_education.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 상세 통계 요약

In [ ]:
# 종합 리포트 생성
summary = pd.DataFrame({
    'Protected Attribute': ['Sex', 'Education Level'],
    'Min Rate': [sex_bias['min_rate'], edu_bias['min_rate']],
    'Max Rate': [sex_bias['max_rate'], edu_bias['max_rate']],
    '4/5 Rule Ratio': [sex_bias['four_fifths_ratio'], edu_bias['four_fifths_ratio']],
    'Pass (>=0.8)': [sex_bias['is_fair'], edu_bias['is_fair']]
})

summary['Disparity (%)'] = (1 - summary['4/5 Rule Ratio']) * 100

print("\n" + "="*80)
print("라벨링 편향 평가 종합 리포트")
print("="*80)
print(summary.to_string(index=False))
print("="*80)

## 6. 결론 및 개선 방향

### 평가 결과 해석

1. **성별(Sex) 편향**
   - 남성과 여성의 고소득(>50K) 라벨 비율에 큰 차이가 존재
   - 4/5 Rule 실패 시 → 성별에 따른 라벨링 편향 존재
   - 역사적/사회적 불평등이 데이터에 반영된 것으로 해석

2. **교육 수준(Education Level) 편향**
   - 고학력과 저학력의 고소득 라벨 비율 차이
   - 4/5 Rule 실패 시 → 교육 수준에 따른 라벨링 편향 존재
   - 합리적인 차이일 수 있으나, 과도한 차이는 문제

### 개선 방향

#### 1. 데이터 수집 단계
- **균형 잡힌 샘플링**: 보호 변수별로 유사한 비율의 긍정/부정 샘플 수집
- **라벨링 가이드라인**: 명확한 기준으로 라벨러의 주관적 편향 최소화
- **다양한 라벨러**: 여러 배경의 라벨러를 활용하여 편향 상쇄

#### 2. 데이터 전처리 단계
- **리샘플링(Resampling)**: 과소표현된 그룹의 샘플 증가 (Oversampling)
- **가중치 조정**: 그룹별 샘플 가중치를 조정하여 불균형 완화
- **합성 데이터**: SMOTE 등의 기법으로 소수 그룹 데이터 생성

#### 3. 모델 학습 단계
- **공정성 제약 조건**: 학습 시 공정성 메트릭을 손실 함수에 포함
- **후처리 기법**: 모델 출력을 조정하여 4/5 Rule 만족
- **Fairness-aware ML**: Fairlearn, AIF360 등 공정성 라이브러리 활용

#### 4. 모니터링 및 감사
- **정기적 평가**: 주기적으로 4/5 Rule 체크
- **편향 리포트**: 이해관계자에게 투명하게 공개
- **피드백 루프**: 실제 운영 데이터로 편향 재평가 및 개선

### 윤리적 고려사항

- **과거 편향의 재생산**: 역사적 불평등이 담긴 데이터를 그대로 사용하면 편향이 고착화됨
- **보호 변수의 선택**: 법적으로 보호되는 변수(성별, 인종 등)를 명확히 정의
- **공정성의 다면성**: 4/5 Rule은 하나의 기준일 뿐, 다양한 공정성 개념을 함께 고려

### 참고 자료
- EEOC (미국 고용기회평등위원회) 4/5 Rule 가이드라인
- Fairlearn: https://fairlearn.org/
- AI Fairness 360: https://aif360.mybluemix.net/